# Notebook 05 — Robustification and Adaptive Pricing: Combined Analysis

This notebook unifies notebooks 02 and 03.  For each combination of
willingness-to-pay (`v_scale`), congestion cost (`w`), and uncertainty budget
(`Γ`) the nominal and robust problems are solved **once**.  All four profit
streams are recorded per disruption scenario, eliminating the redundant
re-solves across the two earlier notebooks.

## Four profit streams per scenario

| Label | Solution | Pricing | Description |
|---|---|---|---|
| `a_nom` | Nominal *x* | Adaptive | Full recourse — nominal solution |
| `b_nom` | Nominal *x* | Fixed pre-disruption prices | Price-rigid — nominal solution |
| `a_rob` | Robust *x*  | Adaptive | Full recourse — robust solution |
| `b_rob` | Robust *x*  | Fixed pre-disruption prices | Price-rigid — robust solution |

**Robustification value** = `a_rob − a_nom` (adaptive) or `b_rob − b_nom` (fixed-price)
**Value of Adaptive Pricing (VAP)** = `a_nom − b_nom` (nominal) or `a_rob − b_rob` (robust)
**Combined gain** = `a_rob − b_nom`  — full benefit of being both robust and price-flexible


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import itertools
import time as _time

from rcflp import (
    instancemaker,
    solve_nominal,
    solve_CCG,
    evaluate_second_stage,
    evaluate_fixed_price,
    compute_epsilon_scalar,
    worst_case_disruption,
    no_disruption_scenario,
    sample_disruptions,
    compute_risk_metrics,
)

print('Imports OK.')


## 1. Configuration

In [ ]:
# --- Instance dimensions ---
In, Jn, Rn = 10, 8, 2
Hn         = 2
DATA_PATH  = '../dataset.xlsx'

# --- Baseline parameters (deep-dive) ---
V_SCALE_BASE = 0.75
W_BASE       = 10.0
GAMMA_BASE   = 2

# --- Sensitivity grid ---
V_SCALE_LIST = [0.50, 0.75, 1.00]
W_LIST       = [5.0, 10.0, 20.0]
GAMMA_LIST   = [1, 2, 3]

# --- Solver ---
TOL        = 0.01
TIME_LIMIT = 600

# --- Monte Carlo ---
N_SAMPLES      = 50   # baseline deep-dive
N_SAMPLES_SENS = 30   # sensitivity loop
SEED           = 42

n_combos = len(V_SCALE_LIST) * len(W_LIST) * len(GAMMA_LIST)
print(f'Instance : I={In}, J={Jn}, R={Rn}, Hn={Hn}')
print(f'Baseline : v_scale={V_SCALE_BASE}, w={W_BASE}, Gamma={GAMMA_BASE}')
print(f'Grid     : {n_combos} combos x {N_SAMPLES_SENS} OOS = '
      f'{n_combos * N_SAMPLES_SENS * 4} SOCP solves (+ WC + pre-disruption)')


## 2. Baseline Instance — Detailed Analysis

We solve the baseline instance (v_scale=0.75, w=10, Γ=2) and evaluate all four
profit streams under both the worst-case disruption and a Monte Carlo sample.


In [ ]:
inst  = instancemaker(In, Jn, Rn, V_SCALE_BASE, W_BASE, data_path=DATA_PATH)
nom   = solve_nominal(inst)
x_nom = nom['x_jr']
print(f'Nominal profit: {nom["profit"]:,.1f}  (runtime: {nom["runtime"]:.1f}s)')

ccg   = solve_CCG(inst, GAMMA_BASE, Hn, x_init=x_nom, tol=TOL,
                   time_limit=TIME_LIMIT, verbose=True)
x_rob = ccg['x_jr']
ccg_gap = 100 * (ccg['UB'] - ccg['LB']) / max(abs(ccg['UB']), 1e-9)
print(f'Robust  profit (LB): {ccg["profit_LB"]:,.1f}')
print(f'  converged={ccg["converged"]}, iters={ccg["n_iter"]}, '
      f'gap={ccg_gap:.2f}%, runtime={ccg["runtime"]:.1f}s')

# Pre-disruption prices
eps0      = no_disruption_scenario(inst, Hn)
p_nom     = evaluate_second_stage(inst, x_nom, eps0, Hn)['prices']
p_rob     = evaluate_second_stage(inst, x_rob, eps0, Hn)['prices']
nd_profit_nom = evaluate_second_stage(inst, x_nom, eps0, Hn)['profit']
nd_profit_rob = evaluate_second_stage(inst, x_rob, eps0, Hn)['profit']
print(f'\nNo-disruption profit — nominal x: {nd_profit_nom:,.1f}')
print(f'No-disruption profit — robust  x: {nd_profit_rob:,.1f}')


In [ ]:
eps_wc_nom, _ = worst_case_disruption(inst, x_nom, GAMMA_BASE, Hn)
eps_wc_rob, _ = worst_case_disruption(inst, x_rob, GAMMA_BASE, Hn)

wc_a_nom = evaluate_second_stage(inst, x_nom, eps_wc_nom, Hn)
wc_b_nom = evaluate_fixed_price(inst,  x_nom, p_nom, eps_wc_nom, Hn)
wc_a_rob = evaluate_second_stage(inst, x_rob, eps_wc_rob, Hn)
wc_b_rob = evaluate_fixed_price(inst,  x_rob, p_rob, eps_wc_rob, Hn)

wc_profits = {
    'a_nom': wc_a_nom['profit'], 'b_nom': wc_b_nom['profit'],
    'a_rob': wc_a_rob['profit'], 'b_rob': wc_b_rob['profit'],
}

print('=== Worst-case profits ===')
for k, v in wc_profits.items():
    print(f'  {k}: {v:>10,.1f}')
print()
print(f'  VAP  (nominal x)         : {wc_profits["a_nom"] - wc_profits["b_nom"]:>10,.1f}')
print(f'  VAP  (robust  x)         : {wc_profits["a_rob"] - wc_profits["b_rob"]:>10,.1f}')
print(f'  Robustification (Scen A) : {wc_profits["a_rob"] - wc_profits["a_nom"]:>10,.1f}')
print(f'  Robustification (Scen B) : {wc_profits["b_rob"] - wc_profits["b_nom"]:>10,.1f}')
print(f'  Combined gain (a_rob-b_nom): {wc_profits["a_rob"] - wc_profits["b_nom"]:>10,.1f}')


In [ ]:
scenarios = sample_disruptions(inst, GAMMA_BASE, Hn, N_SAMPLES, SEED)

pa_nom_list, pb_nom_list, pa_rob_list, pb_rob_list = [], [], [], []
for eps in scenarios:
    pa_nom_list.append(evaluate_second_stage(inst, x_nom, eps, Hn)['profit'])
    pb_nom_list.append(evaluate_fixed_price(inst,  x_nom, p_nom, eps, Hn)['profit'])
    pa_rob_list.append(evaluate_second_stage(inst, x_rob, eps, Hn)['profit'])
    pb_rob_list.append(evaluate_fixed_price(inst,  x_rob, p_rob, eps, Hn)['profit'])

pa_nom = np.array(pa_nom_list); pb_nom = np.array(pb_nom_list)
pa_rob = np.array(pa_rob_list); pb_rob = np.array(pb_rob_list)
vap_nom  = pa_nom - pb_nom
vap_rob  = pa_rob - pb_rob
rob_val  = pa_rob - pa_nom
combined = pa_rob - pb_nom

print(f'OOS averages (N={N_SAMPLES}):')
for label, arr in [('a_nom', pa_nom), ('b_nom', pb_nom), ('a_rob', pa_rob), ('b_rob', pb_rob)]:
    print(f'  {label}: mean={np.mean(arr):>8,.1f}  std={np.std(arr):,.1f}  '
          f'min={np.min(arr):,.1f}')
print()
print(f'  Avg VAP nom     : {np.mean(vap_nom):>8,.1f}')
print(f'  Avg VAP rob     : {np.mean(vap_rob):>8,.1f}')
print(f'  Avg Rob value   : {np.mean(rob_val):>8,.1f}')
print(f'  Avg Combined    : {np.mean(combined):>8,.1f}')


In [ ]:
rm_rob   = compute_risk_metrics(pa_nom_list, pa_rob_list)   # nom vs rob (ScenA)
rm_vap_n = compute_risk_metrics(pb_nom_list, pa_nom_list)   # fixed vs adaptive (nominal x)
rm_vap_r = compute_risk_metrics(pb_rob_list, pa_rob_list)   # fixed vs adaptive (robust x)

METRIC_LABELS = {
    'mean_profit': 'Mean profit', 'min_profit': 'Min profit',
    'pct5_profit': '5th pct',    'cvar5':       'CVaR 5%',
    'cvar10':      'CVaR 10%',   'prob_loss':   'P(loss)',
    'mean_regret': 'Mean regret','max_regret':  'Max regret',
}

def _print_rm(rm, lbl_a, lbl_b, title):
    print(f'\n{title}')
    print(f'  {"Metric":<18} {lbl_a:>12} {lbl_b:>12}')
    print('  ' + '-' * 44)
    for k, label in METRIC_LABELS.items():
        va, vb = rm['nominal'][k], rm['robust'][k]
        fmt = '.1%' if k == 'prob_loss' else ',.1f'
        print(f'  {label:<18} {va:>12{fmt}} {vb:>12{fmt}}')

_print_rm(rm_rob,   'Nominal x', 'Robust x',  '=== Robustification (ScenA — adaptive) ===')
_print_rm(rm_vap_n, 'Fixed-px',  'Adaptive',  '=== VAP — Nominal x ===')
_print_rm(rm_vap_r, 'Fixed-px',  'Adaptive',  '=== VAP — Robust x ===')


## 3. Baseline Visualisation

In [ ]:
STREAM_COLORS = {
    'a_nom': '#2196F3', 'b_nom': '#FF5722',
    'a_rob': '#4CAF50', 'b_rob': '#FF9800',
}
STREAM_LABELS = {
    'a_nom': 'Nom + Adaptive', 'b_nom': 'Nom + Fixed-px',
    'a_rob': 'Rob + Adaptive', 'b_rob': 'Rob + Fixed-px',
}
streams = ['a_nom', 'b_nom', 'a_rob', 'b_rob']
oos_arrays = [pa_nom, pb_nom, pa_rob, pb_rob]
wc_vals   = [wc_profits[s] for s in streams]

fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# [0,0] Worst-case grouped bar ─────────────────────────────────────────────
ax = axes[0, 0]
x_pos = np.arange(4)
bars  = ax.bar(x_pos, wc_vals, color=[STREAM_COLORS[s] for s in streams],
               edgecolor='k', linewidth=0.7, alpha=0.88, width=0.6)
ax.set_xticks(x_pos)
ax.set_xticklabels([STREAM_LABELS[s] for s in streams], rotation=10, ha='right', fontsize=9)
ax.set_ylabel('Profit', fontsize=11)
ax.set_title('Worst-case Profit per Stream', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', linestyle='--', alpha=0.4)
# VAP annotations
for idx_a, idx_b, col, label in [
    (0, 1, '#2196F3', 'VAP\nnominal'),
    (2, 3, '#4CAF50', 'VAP\nrobust'),
]:
    va, vb = wc_vals[idx_a], wc_vals[idx_b]
    mid = (va + vb) / 2
    ax.annotate('', xy=(idx_a, va), xytext=(idx_b, vb),
                arrowprops=dict(arrowstyle='<->', color=col, lw=1.8))
    ax.text((idx_a + idx_b)/2, mid, f'{label}\n{va-vb:,.0f}',
            ha='center', va='center', fontsize=8, color=col,
            bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.85))

# [0,1] Violin + mean diamonds ─────────────────────────────────────────────
ax = axes[0, 1]
vp = ax.violinplot(oos_arrays, positions=range(4), showmedians=True, showextrema=True)
for patch, s in zip(vp['bodies'], streams):
    patch.set_facecolor(STREAM_COLORS[s]); patch.set_alpha(0.72)
vp['cmedians'].set_colors('black'); vp['cmedians'].set_linewidth(2)
for i, arr in enumerate(oos_arrays):
    ax.scatter(i, np.mean(arr), marker='D', s=55, color='white',
               edgecolor='black', zorder=5, linewidth=1.5)
ax.set_xticks(range(4))
ax.set_xticklabels([STREAM_LABELS[s] for s in streams], rotation=10, ha='right', fontsize=9)
ax.set_ylabel('Profit', fontsize=11)
ax.set_title(f'OOS Profit Distribution  (N={N_SAMPLES})', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='y', linestyle='--', alpha=0.4)

# [0,2] Empirical CDF ──────────────────────────────────────────────────────
ax = axes[0, 2]
for s, arr in zip(streams, oos_arrays):
    sx = np.sort(arr)
    cdf = np.arange(1, len(arr)+1) / len(arr)
    ax.step(sx, cdf, color=STREAM_COLORS[s], label=STREAM_LABELS[s], lw=2, where='post')
    ax.axvline(np.percentile(arr, 5), color=STREAM_COLORS[s], lw=0.8,
               linestyle=':', alpha=0.7)
ax.axvline(0, color='black', linestyle=':', lw=1, alpha=0.5)
ax.set_xlabel('Profit', fontsize=11)
ax.set_ylabel('Cumulative probability', fontsize=11)
ax.set_title('Empirical CDF (dotted = 5th pct)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8); ax.grid(linestyle='--', alpha=0.4); ax.set_ylim(0, 1.02)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# [1,0] Cost breakdown horizontal stacked bars ────────────────────────────
ax = axes[1, 0]
bd_list   = [wc_a_nom['breakdown'], wc_b_nom['breakdown'],
             wc_a_rob['breakdown'], wc_b_rob['breakdown']]
bd_labels = ['Nom+Adapt', 'Nom+Fixed', 'Rob+Adapt', 'Rob+Fixed']
comp_cfg  = [('revenue',    '#43A047', 'Revenue'),
             ('transport',  '#EF5350', '− Transport'),
             ('fixed_cost', '#5C6BC0', '− Fixed cost'),
             ('congestion', '#FF8F00', '− Congestion')]
for i, bd in enumerate(bd_list):
    left_p, left_n = 0.0, 0.0
    for comp, col, lbl in comp_cfg:
        val = bd[comp] if comp == 'revenue' else -bd[comp]
        if val >= 0:
            ax.barh(i, val, left=left_p, color=col, alpha=0.85, height=0.55,
                    label=lbl if i == 0 else '')
            left_p += val
        else:
            ax.barh(i, val, left=left_n, color=col, alpha=0.85, height=0.55)
            left_n += val
ax.axvline(0, color='black', lw=0.8)
ax.set_yticks(range(4)); ax.set_yticklabels(bd_labels, fontsize=10)
ax.set_xlabel('Value', fontsize=11)
ax.set_title('Cost Breakdown (worst-case scenario)', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='lower right')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(axis='x', linestyle='--', alpha=0.4)

# [1,1] Robustification value vs VAP scatter (per OOS scenario) ───────────
ax = axes[1, 1]
sc = ax.scatter(rob_val, vap_rob, c=combined, cmap='RdYlGn',
                s=65, edgecolors='k', linewidth=0.5, alpha=0.88, zorder=3)
plt.colorbar(sc, ax=ax, label='Combined gain  (a_rob − b_nom)')
ax.axhline(0, color='gray', lw=0.8, linestyle='--')
ax.axvline(0, color='gray', lw=0.8, linestyle='--')
ax.set_xlabel('Robustification value  (a_rob − a_nom)', fontsize=11)
ax.set_ylabel('VAP for robust x  (a_rob − b_rob)', fontsize=11)
ax.set_title('Robustification vs VAP per Scenario', fontsize=12, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(linestyle='--', alpha=0.3)

# [1,2] Profit ladder (all 4 streams sorted by b_nom) ────────────────────
ax = axes[1, 2]
idx = np.argsort(pb_nom)
xr  = np.arange(N_SAMPLES)
ax.fill_between(xr, pb_nom[idx], pa_nom[idx], alpha=0.22,
                color='#2196F3', label='VAP (nom x)')
ax.fill_between(xr, pa_nom[idx], pa_rob[idx], alpha=0.22,
                color='#4CAF50', label='Robustification')
ax.plot(xr, pb_nom[idx], color='#FF5722', lw=1.8, label='b_nom')
ax.plot(xr, pa_nom[idx], color='#2196F3', lw=1.8, label='a_nom')
ax.plot(xr, pb_rob[idx], color='#FF9800', lw=1.4, linestyle='--',
        label='b_rob', alpha=0.8)
ax.plot(xr, pa_rob[idx], color='#4CAF50', lw=1.8, label='a_rob')
ax.axhline(0, color='black', lw=0.7, linestyle=':')
ax.set_xlabel('Scenario (sorted by b_nom profit)', fontsize=11)
ax.set_ylabel('Profit', fontsize=11)
ax.set_title('Profit Ladder Across Scenarios', fontsize=12, fontweight='bold')
ax.legend(fontsize=8, ncol=2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.grid(linestyle='--', alpha=0.3)

fig.suptitle(
    f'Baseline Analysis  |  v_scale={V_SCALE_BASE}, w={W_BASE}, '
    f'Γ={GAMMA_BASE}  |  {N_SAMPLES} OOS scenarios',
    fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('05_fig1_baseline.pdf', bbox_inches='tight')
plt.savefig('05_fig1_baseline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 05_fig1_baseline.pdf/.png')


## 4. Sensitivity Analysis

Sweep all 27 combinations of (v_scale, w, Γ).  For each combination:
1. Solve nominal + CCG (once)
2. Evaluate all four streams under worst-case disruption
3. Sample `N_SAMPLES_SENS` OOS scenarios and evaluate all four streams per scenario
4. Compute statistics for every derived quantity (VAP, robustification value, combined gain)


In [ ]:
def _stats(arr):
    arr = np.asarray(arr, dtype=float)
    n = len(arr)
    k5  = max(1, int(np.ceil(0.05 * n)))
    k10 = max(1, int(np.ceil(0.10 * n)))
    s   = np.sort(arr)
    return dict(
        mean=float(np.mean(arr)), std=float(np.std(arr)),
        min=float(s[0]),          pct5=float(np.percentile(arr, 5)),
        pct25=float(np.percentile(arr, 25)), median=float(np.median(arr)),
        pct75=float(np.percentile(arr, 75)), max=float(s[-1]),
        cvar5=float(np.mean(s[:k5])), cvar10=float(np.mean(s[:k10])),
        prob_loss=float(np.mean(arr < 0)), prob_pos=float(np.mean(arr > 0)),
    )

sens_rows, oos_rows = [], []
combo_list = list(itertools.product(V_SCALE_LIST, W_LIST, GAMMA_LIST))
n_total    = len(combo_list)

for run_idx, (v, w, gam) in enumerate(combo_list, 1):
    t0 = _time.time()
    print(f'[{run_idx:2d}/{n_total}]  v={v}  w={w:4.0f}  Γ={gam}', end='  ', flush=True)

    inst_s  = instancemaker(In, Jn, Rn, v, w, data_path=DATA_PATH)
    nom_s   = solve_nominal(inst_s)
    x_nom_s = nom_s['x_jr']
    ccg_s   = solve_CCG(inst_s, gam, Hn, x_init=x_nom_s, tol=TOL, time_limit=TIME_LIMIT)
    x_rob_s = ccg_s['x_jr']

    eps0_s       = no_disruption_scenario(inst_s, Hn)
    nd_res_nom_s = evaluate_second_stage(inst_s, x_nom_s, eps0_s, Hn)
    nd_res_rob_s = evaluate_second_stage(inst_s, x_rob_s, eps0_s, Hn)
    p_nom_s      = nd_res_nom_s['prices']
    p_rob_s      = nd_res_rob_s['prices']

    eps_wc_n_s, _ = worst_case_disruption(inst_s, x_nom_s, gam, Hn)
    eps_wc_r_s, _ = worst_case_disruption(inst_s, x_rob_s, gam, Hn)

    wc_an = evaluate_second_stage(inst_s, x_nom_s, eps_wc_n_s, Hn)['profit']
    wc_bn = evaluate_fixed_price(inst_s,  x_nom_s, p_nom_s, eps_wc_n_s, Hn)['profit']
    wc_ar = evaluate_second_stage(inst_s, x_rob_s, eps_wc_r_s, Hn)['profit']
    wc_br = evaluate_fixed_price(inst_s,  x_rob_s, p_rob_s, eps_wc_r_s, Hn)['profit']

    scen_s = sample_disruptions(inst_s, gam, Hn, N_SAMPLES_SENS, SEED)
    pan_s, pbn_s, par_s, pbr_s = [], [], [], []
    for k, eps_s in enumerate(scen_s):
        pan = evaluate_second_stage(inst_s, x_nom_s, eps_s, Hn)['profit']
        pbn = evaluate_fixed_price(inst_s,  x_nom_s, p_nom_s, eps_s, Hn)['profit']
        par = evaluate_second_stage(inst_s, x_rob_s, eps_s, Hn)['profit']
        pbr = evaluate_fixed_price(inst_s,  x_rob_s, p_rob_s, eps_s, Hn)['profit']
        pan_s.append(pan); pbn_s.append(pbn)
        par_s.append(par); pbr_s.append(pbr)
        oos_rows.append(dict(
            v_scale=v, w=w, gamma=gam, scenario_k=k,
            profit_a_nom=pan, profit_b_nom=pbn,
            profit_a_rob=par, profit_b_rob=pbr,
            vap_nom=pan - pbn, vap_rob=par - pbr,
            rob_val_a=par - pan, rob_val_b=pbr - pbn,
            combined=par - pbn,
        ))

    pan_s  = np.array(pan_s); pbn_s  = np.array(pbn_s)
    par_s  = np.array(par_s); pbr_s  = np.array(pbr_s)
    vap_n  = pan_s - pbn_s;   vap_r  = par_s - pbr_s
    rob_a  = par_s - pan_s;   rob_b  = pbr_s - pbn_s
    comb   = par_s - pbn_s

    elapsed = _time.time() - t0
    print(f'done ({elapsed:.0f}s) | avg_rob={np.mean(rob_a):,.0f}  avg_vap={np.mean(vap_r):,.0f}')

    row = dict(
        v_scale=v, w=w, gamma=gam,
        nom_profit=nom_s['profit'], nom_runtime=nom_s['runtime'],
        rob_profit_LB=ccg_s['profit_LB'], rob_runtime=ccg_s['runtime'],
        ccg_n_iter=ccg_s['n_iter'], ccg_converged=int(ccg_s['converged']),
        ccg_gap_pct=100*(ccg_s['UB']-ccg_s['LB'])/max(abs(ccg_s['UB']),1e-9),
        nd_nom=nd_res_nom_s['profit'], nd_rob=nd_res_rob_s['profit'],
        wc_a_nom=wc_an, wc_b_nom=wc_bn, wc_a_rob=wc_ar, wc_b_rob=wc_br,
        wc_vap_nom=wc_an-wc_bn,  wc_vap_rob=wc_ar-wc_br,
        wc_rob_val_a=wc_ar-wc_an, wc_rob_val_b=wc_br-wc_bn,
        wc_combined=wc_ar-wc_bn,
    )
    for name, arr in [('a_nom',pan_s),('b_nom',pbn_s),('a_rob',par_s),('b_rob',pbr_s),
                      ('vap_nom',vap_n),('vap_rob',vap_r),
                      ('rob_val_a',rob_a),('rob_val_b',rob_b),('combined',comb)]:
        for m, val in _stats(arr).items():
            row[f'{name}_{m}'] = val
    sens_rows.append(row)

df_sens = pd.DataFrame(sens_rows)
df_oos  = pd.DataFrame(oos_rows)
print(f'\nDone. Summary: {df_sens.shape}  |  OOS raw: {df_oos.shape}')


## 5. Save Results to Excel

In [ ]:
EXCEL_PATH = '05_sensitivity_results.xlsx'

pivot_specs = [
    ('wc_rob_val_a',  'WC_RobVal_ScenA'),
    ('wc_vap_nom',    'WC_VAP_Nom'),
    ('wc_vap_rob',    'WC_VAP_Rob'),
    ('wc_combined',   'WC_Combined'),
    ('rob_val_a_mean','Avg_RobVal_ScenA'),
    ('vap_nom_mean',  'Avg_VAP_Nom'),
    ('vap_rob_mean',  'Avg_VAP_Rob'),
    ('combined_mean', 'Avg_Combined'),
    ('a_rob_cvar5',   'ScenA_Rob_CVaR5'),
    ('a_nom_cvar5',   'ScenA_Nom_CVaR5'),
    ('b_rob_cvar5',   'ScenB_Rob_CVaR5'),
    ('b_nom_cvar5',   'ScenB_Nom_CVaR5'),
    ('combined_cvar5','Combined_CVaR5'),
    ('vap_rob_cvar5', 'VAP_Rob_CVaR5'),
    ('vap_nom_cvar5', 'VAP_Nom_CVaR5'),
    ('a_rob_prob_loss','ScenA_Rob_ProbLoss'),
    ('a_nom_prob_loss','ScenA_Nom_ProbLoss'),
    ('rob_val_a_cvar5','RobVal_CVaR5'),
]

with pd.ExcelWriter(EXCEL_PATH, engine='openpyxl') as writer:
    df_sens.to_excel(writer, sheet_name='Summary',  index=False)
    df_oos.to_excel( writer, sheet_name='OOS_Raw',  index=False)
    for col, sheet in pivot_specs:
        piv = df_sens.pivot_table(
            index='w', columns=['v_scale', 'gamma'], values=col)
        piv.to_excel(writer, sheet_name=sheet)

print(f'Saved to {EXCEL_PATH}')
print(f'  Sheets: Summary, OOS_Raw + {len(pivot_specs)} pivot sheets')


## 6. Sensitivity Visualisations

In [ ]:
# ── Figure 2: 2×2 Value Matrix — how the four streams rank across Gamma ──────
# At v=0.75, w=10: one subplot per Gamma value, showing the 2x2 grid
# (Nominal/Robust) x (Fixed-px/Adaptive) mean profit as a tile map.

sub = df_sens[(df_sens['v_scale'] == 0.75) & (df_sens['w'] == 10.0)].sort_values('gamma')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, (_, row) in zip(axes, sub.iterrows()):
    gam = int(row['gamma'])
    matrix = np.array([
        [row['b_nom_mean'], row['a_nom_mean']],  # Nominal: Fixed | Adaptive
        [row['b_rob_mean'], row['a_rob_mean']],  # Robust:  Fixed | Adaptive
    ])
    vmin, vmax = matrix.min() * 0.95, matrix.max() * 1.05
    im = ax.imshow(matrix, cmap='RdYlGn', vmin=vmin, vmax=vmax, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.8)

    for i in range(2):
        for j in range(2):
            ax.text(j, i, f'{matrix[i,j]:,.0f}', ha='center', va='center',
                    fontsize=13, fontweight='bold',
                    color='white' if matrix[i,j] < (vmin+vmax)/2 else 'black')

    # Annotations for VAP and robustification
    vap_n   = matrix[0,1] - matrix[0,0]
    vap_r   = matrix[1,1] - matrix[1,0]
    rob_a   = matrix[1,1] - matrix[0,1]
    rob_b   = matrix[1,0] - matrix[0,0]
    comb    = matrix[1,1] - matrix[0,0]

    ax.set_xticks([0, 1]); ax.set_xticklabels(['Fixed-price', 'Adaptive'], fontsize=11)
    ax.set_yticks([0, 1]); ax.set_yticklabels(['Nominal x', 'Robust x'], fontsize=11)
    ax.set_title(
        f'Γ = {gam}  |  mean profit\n'
        f'VAP nom={vap_n:,.0f}  VAP rob={vap_r:,.0f}\n'
        f'Rob(A)={rob_a:,.0f}  Rob(B)={rob_b:,.0f}  Combined={comb:,.0f}',
        fontsize=9.5)

fig.suptitle('Value Matrix (v=0.75, w=10)  —  mean OOS profit per (solution × pricing) cell',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('05_fig2_value_matrix.pdf', bbox_inches='tight')
plt.savefig('05_fig2_value_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 05_fig2_value_matrix.pdf/.png')


In [ ]:
# ── Figure 3: 2×3 heatmap grid at v_scale = 0.75 (w × Gamma) ───────────────
sub075 = df_sens[df_sens['v_scale'] == 0.75].copy()

metrics_hm = [
    ('rob_val_a_mean', 'Avg Robustification value\n(a_rob − a_nom)'),
    ('vap_rob_mean',   'Avg VAP — Robust x\n(a_rob − b_rob)'),
    ('combined_mean',  'Avg Combined gain\n(a_rob − b_nom)'),
    ('a_rob_cvar5',    'CVaR 5% — a_rob\n(risk-adjusted robust profit)'),
    ('vap_rob_cvar5',  'CVaR 5% of VAP — Robust x'),
    ('a_rob_prob_loss','P(loss) — a_rob'),
]

fig, axes = plt.subplots(2, 3, figsize=(17, 10))
axes = axes.flatten()

for ax, (col, title) in zip(axes, metrics_hm):
    piv = sub075.pivot(index='w', columns='gamma', values=col)
    cmap = 'RdYlGn' if 'prob_loss' not in col else 'RdYlGn_r'
    im = ax.imshow(piv.values, cmap=cmap, aspect='auto', origin='lower')
    plt.colorbar(im, ax=ax)
    ax.set_xticks(range(len(GAMMA_LIST)))
    ax.set_xticklabels([f'Γ={g}' for g in GAMMA_LIST], fontsize=10)
    ax.set_yticks(range(len(W_LIST)))
    ax.set_yticklabels([f'w={w}' for w in W_LIST], fontsize=10)
    ax.set_xlabel('Uncertainty budget', fontsize=10)
    ax.set_ylabel('Congestion cost', fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    for i in range(len(W_LIST)):
        for j in range(len(GAMMA_LIST)):
            val = piv.values[i, j]
            fmt = f'{val:.1%}' if 'prob_loss' in col else f'{val:,.0f}'
            ax.text(j, i, fmt, ha='center', va='center',
                    fontsize=10, fontweight='bold', color='black')

fig.suptitle('Sensitivity Heatmaps  (v_scale = 0.75)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('05_fig3_heatmaps.pdf', bbox_inches='tight')
plt.savefig('05_fig3_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 05_fig3_heatmaps.pdf/.png')


In [ ]:
# ── Figure 4: 2×2 line plots — sensitivity vs Γ for different w (v=0.75) ────
sub075 = df_sens[df_sens['v_scale'] == 0.75]
W_COLORS = {5.0: 'royalblue', 10.0: 'tomato', 20.0: 'forestgreen'}

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

panels = [
    (axes[0,0],
     [('rob_val_a_mean', 'Robustification', '-o'),
      ('rob_val_b_mean', 'Rob. (fixed-px)', '--s')],
     'Robustification Value vs Γ',
     'Value (a_rob − a_nom  or  b_rob − b_nom)'),
    (axes[0,1],
     [('vap_nom_mean', 'VAP nominal x', '-o'),
      ('vap_rob_mean', 'VAP robust x',  '--s')],
     'Value of Adaptive Pricing vs Γ',
     'VAP  (a − b)'),
    (axes[1,0],
     [('a_rob_cvar5',  'CVaR5 a_rob', '-o'),
      ('a_nom_cvar5',  'CVaR5 a_nom', '--s')],
     'CVaR 5%  (adaptive pricing) vs Γ',
     'CVaR 5% of profit'),
    (axes[1,1],
     [('combined_mean',  'Combined mean', '-o'),
      ('combined_cvar5', 'Combined CVaR5', '--s')],
     'Combined Gain (a_rob − b_nom) vs Γ',
     'Combined gain'),
]

for ax, series_cfg, title, ylabel in panels:
    for col, label, style in series_cfg:
        for w_val in W_LIST:
            g = sub075[sub075['w'] == w_val].sort_values('gamma')
            lw = 2 if '-o' in style else 1.5
            ax.plot(g['gamma'], g[col], style, color=W_COLORS[w_val],
                    lw=lw, ms=7,
                    label=f'{label}, w={w_val}' if w_val == W_LIST[0] else f'w={w_val}')
    ax.set_xlabel('Uncertainty budget Γ', fontsize=11)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xticks(GAMMA_LIST)
    ax.grid(linestyle='--', alpha=0.4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Build a shared legend
handles = []
for col, label, style in panels[0][1]:
    handles.append(Line2D([0],[0], linestyle=style[1], marker=style[-1],
                          color='gray', lw=2, label=label))
for w_val, col in W_COLORS.items():
    handles.append(mpatches.Patch(color=col, label=f'w = {w_val}'))
fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=9,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle('Sensitivity Line Plots  (v_scale = 0.75)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('05_fig4_sensitivity_lines.pdf', bbox_inches='tight')
plt.savefig('05_fig4_sensitivity_lines.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 05_fig4_sensitivity_lines.pdf/.png')


In [ ]:
# ── Figure 5: Risk landscape — how risk metrics of a_rob change across ───────
# all three v_scale values as Γ increases (at w = 10)

sub_w10 = df_sens[df_sens['w'] == 10.0]
V_COLORS = {0.50: '#9C27B0', 0.75: '#FF6F00', 1.00: '#00796B'}

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

risk_panels = [
    ('a_rob_min',      'Worst realised profit  (min)', '#4CAF50'),
    ('a_rob_pct5',     '5th percentile profit (VaR 95%)', '#4CAF50'),
    ('a_rob_cvar5',    'CVaR 5%  (mean of worst 5%)', '#4CAF50'),
    ('rob_val_a_mean', 'Mean robustification value', '#2196F3'),
    ('vap_rob_mean',   'Mean VAP (robust x)', '#FF5722'),
    ('a_rob_prob_loss','P(loss < 0)  — robust+adaptive', '#9C27B0'),
]

for ax, (col, title, ref_col) in zip(axes, risk_panels):
    for v_val in V_SCALE_LIST:
        g = sub_w10[sub_w10['v_scale'] == v_val].sort_values('gamma')
        ax.plot(g['gamma'], g[col], '-o', color=V_COLORS[v_val],
                lw=2, ms=8, label=f'v_scale={v_val}')

    ax.set_xlabel('Uncertainty budget Γ', fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xticks(GAMMA_LIST)
    ax.grid(linestyle='--', alpha=0.4)
    if 'prob_loss' in col:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.1%}'))
        ax.axhline(0, color='black', lw=0.8, linestyle=':')
    else:
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.legend(fontsize=9)

fig.suptitle('Risk Landscape of Robust + Adaptive Solution  (w = 10)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('05_fig5_risk_landscape.pdf', bbox_inches='tight')
plt.savefig('05_fig5_risk_landscape.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 05_fig5_risk_landscape.pdf/.png')


In [ ]:
# ── Figure 6: Box plots from raw OOS data — profit distribution per stream ───
# One subplot per v_scale; x-axis = Gamma; four box traces per Gamma value

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)

stream_cfg = [
    ('profit_b_nom', '#FF5722', 'b_nom'),
    ('profit_a_nom', '#2196F3', 'a_nom'),
    ('profit_b_rob', '#FF9800', 'b_rob'),
    ('profit_a_rob', '#4CAF50', 'a_rob'),
]

for ax, v_val in zip(axes, V_SCALE_LIST):
    sub_v = df_oos[(df_oos['v_scale'] == v_val) & (df_oos['w'] == 10.0)]
    positions, data_by_pos, colors_by_pos = [], [], []
    x_tick_pos, x_tick_labels = [], []
    gam_gap = 1.2
    stream_gap = 0.25
    for gi, gam in enumerate(GAMMA_LIST):
        sub_g = sub_v[sub_v['gamma'] == gam]
        gam_center = gi * gam_gap * (len(stream_cfg) * stream_gap + 0.4)
        for si, (col, color, label) in enumerate(stream_cfg):
            pos = gam_center + si * stream_gap
            positions.append(pos)
            data_by_pos.append(sub_g[col].values)
            colors_by_pos.append(color)
        x_tick_pos.append(gam_center + (len(stream_cfg)-1)*stream_gap/2)
        x_tick_labels.append(f'Γ={gam}')

    bp = ax.boxplot(data_by_pos, positions=positions, widths=0.2,
                    patch_artist=True, notch=False,
                    medianprops=dict(color='black', lw=2),
                    whiskerprops=dict(lw=1), capprops=dict(lw=1),
                    flierprops=dict(marker='o', ms=3, alpha=0.5))
    for patch, col in zip(bp['boxes'], colors_by_pos):
        patch.set_facecolor(col); patch.set_alpha(0.75)

    ax.set_xticks(x_tick_pos); ax.set_xticklabels(x_tick_labels, fontsize=11)
    ax.set_ylabel('Profit', fontsize=11)
    ax.set_title(f'v_scale = {v_val}  (w = 10)', fontsize=12, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.axhline(0, color='black', lw=0.7, linestyle=':')

legend_handles = [mpatches.Patch(color=c, label=lbl, alpha=0.8)
                  for _, c, lbl in stream_cfg]
fig.legend(handles=legend_handles, loc='lower center', ncol=4,
           fontsize=10, bbox_to_anchor=(0.5, -0.06))

fig.suptitle('OOS Profit Distributions by Solution, Pricing, and Γ  (w = 10)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('05_fig6_boxplots.pdf', bbox_inches='tight')
plt.savefig('05_fig6_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 05_fig6_boxplots.pdf/.png')
